# MyDigitalTwin — Spotify
**Notebook — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `data/raw/SPOTIFY/StreamingHistory_music_0-7.json` — historique d'écoute (~95k événements)
- `data/raw/SPOTIFY/YourLibrary.json` — titres likés (~1 500 tracks, signal d'identité fort)
- `data/raw/SPOTIFY/Playlist1.json` — 31 playlists personnelles
- `data/raw/SPOTIFY/YourSoundCapsule.json` — snapshots hebdomadaires avec **genres** (unique source de genre)

Outputs :
- `data/parquet/spotify_streams.parquet` — écoutes nettoyées (>30s) avec features temporelles
- `data/parquet/spotify_library.parquet` — titres sauvegardés (identité musicale)
- `data/parquet/spotify_playlists.parquet` — tracks de toutes les playlists
- `data/parquet/spotify_sound_capsule.parquet` — genres par semaine (pour K-Means)

## Objectifs ML
| Axe | Source | Signal |
|---|---|---|
| ALS (Oracle) | streams + library + playlists | poids 1 / 3 / 2 |
| K-Means (Dashboard) | streams + sound capsule | heure, jour, durée, **genre** |

## 0. Initialisation Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Spotify") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

Spark version : 3.5.5


26/03/31 20:00:36 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Streaming History — Ingestion

8 fichiers JSON chronologiques (mars 2025 → mars 2026).  
Chaque entrée = un événement d'écoute avec `endTime`, `artistName`, `trackName`, `msPlayed`.

In [2]:
STREAMING_FILES  = [
    f"../../data/raw/SPOTIFY/StreamingHistory_music_{i}.json"
    for i in range(8)
]

LIBRARY_PATH      = "../../../data/raw/SPOTIFY/YourLibrary.json"
PLAYLIST_PATH     = "../../../data/raw/SPOTIFY/Playlist1.json"
SOUND_CAPSULE_PATH = "../../../data/raw/SPOTIFY/YourSoundCapsule.json"

OUT_STREAMS       = "../../data/parquet/spotify_streams.parquet"
OUT_LIBRARY       = "../../data/parquet/spotify_library.parquet"
OUT_PLAYLISTS     = "../../data/parquet/spotify_playlists.parquet"
OUT_SOUND_CAPSULE = "../../data/parquet/spotify_sound_capsule.parquet"

# Lecture des 8 fichiers en un seul DataFrame
df_streams_raw = spark.read \
    .option("multiLine", "true") \
    .json(STREAMING_FILES)

print(f"Lignes brutes (toutes écoutes) : {df_streams_raw.count():,}")
df_streams_raw.printSchema()
df_streams_raw.show(5, truncate=50)

Lignes brutes (toutes écoutes) : 75,786
root
 |-- artistName: string (nullable = true)
 |-- endTime: string (nullable = true)
 |-- msPlayed: long (nullable = true)
 |-- trackName: string (nullable = true)

+----------+----------------+--------+----------------------------+
|artistName|         endTime|msPlayed|                   trackName|
+----------+----------------+--------+----------------------------+
|Guy2Bezbar|2025-03-29 17:55|   65940|     Belly (feat. Rick Ross)|
|   KAROL G|2025-03-30 17:51|    1555|Si Antes Te Hubiera Conocido|
|     Rim'K|2025-03-30 17:52|   10240|                         Run|
|       L2B|2025-03-30 17:52|   16161|                     Pélican|
|       L2B|2025-03-30 17:55|  165653|                     Pélican|
+----------+----------------+--------+----------------------------+
only showing top 5 rows



## 2. Streaming History — Exploration & Data Quality

In [3]:
print("=== Valeurs nulles ===")
df_streams_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_streams_raw.columns
]).show()

print("\n=== Période couverte ===")
df_streams_raw.agg(
    F.min("endTime").alias("premier"),
    F.max("endTime").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts")
).show(truncate=False)

=== Valeurs nulles ===
+----------+-------+--------+---------+
|artistName|endTime|msPlayed|trackName|
+----------+-------+--------+---------+
|         0|      0|       0|        0|
+----------+-------+--------+---------+


=== Période couverte ===


+----------------+----------------+------------------+----------------+
|premier         |dernier         |artistes_distincts|titres_distincts|
+----------------+----------------+------------------+----------------+
|2025-03-29 17:55|2026-03-30 23:53|5348              |13772           |
+----------------+----------------+------------------+----------------+



In [4]:
print("=== Distribution msPlayed (ms) ===")
df_streams_raw.select(
    F.min("msPlayed").alias("min_ms"),
    F.max("msPlayed").alias("max_ms"),
    F.mean("msPlayed").alias("mean_ms"),
    F.percentile_approx("msPlayed", 0.5).alias("median_ms")
).show()

print("\n=== Écoutes < 30s (skip) vs >= 30s ===")
df_streams_raw.groupBy(
    F.when(F.col("msPlayed") < 30000, "< 30s (skip)").otherwise(">= 30s").alias("categorie")
).count().orderBy("categorie").show()

=== Distribution msPlayed (ms) ===


+------+------+-----------------+---------+
|min_ms|max_ms|          mean_ms|median_ms|
+------+------+-----------------+---------+
|     0|811077|75851.98373050432|    17777|
+------+------+-----------------+---------+


=== Écoutes < 30s (skip) vs >= 30s ===
+------------+-----+
|   categorie|count|
+------------+-----+
|< 30s (skip)|41814|
|      >= 30s|33972|
+------------+-----+



## 3. Streaming History — Nettoyage

Règles appliquées :
- **Filtre >30s** : `msPlayed >= 30 000` — élimine les skips et les pré-écoutes
- **Parse `endTime`** : format `"YYYY-MM-DD HH:MM"` → timestamp Spark
- **Features temporelles** : heure, jour de la semaine, mois, année
- **`minutes_played`** : conversion ms → minutes pour lisibilité

In [5]:
# Filtre écoutes réelles (>30s)
df_streams = df_streams_raw.filter(F.col("msPlayed") >= 30000)

print(f"Lignes après filtre >30s : {df_streams.count():,}")

# Parse endTime "YYYY-MM-DD HH:MM" → timestamp
df_streams = df_streams.withColumn(
    "listen_ts",
    F.to_timestamp(F.col("endTime"), "yyyy-MM-dd HH:mm")
)

# Features temporelles
df_streams = df_streams \
    .withColumn("listen_year",    F.year("listen_ts")) \
    .withColumn("listen_month",   F.date_format("listen_ts", "yyyy-MM")) \
    .withColumn("listen_hour",    F.hour("listen_ts")) \
    .withColumn("listen_weekday", F.dayofweek("listen_ts")) \
    .withColumn("listen_week",    F.weekofyear("listen_ts")) \
    .withColumn("minutes_played", F.round(F.col("msPlayed") / 60000.0, 2))

# Plage nuit (22h–5h) pour K-Means
df_streams = df_streams.withColumn(
    "is_night",
    F.when((F.col("listen_hour") >= 22) | (F.col("listen_hour") <= 5), True).otherwise(False)
)

# Poids ALS : écoute standard = 1.0
df_streams = df_streams.withColumn("interaction_weight", F.lit(1.0))

df_streams.select(
    "artistName", "trackName", "listen_ts", "listen_hour", "listen_weekday", "minutes_played", "is_night"
).show(8, truncate=40)

Lignes après filtre >30s : 33,972
+----------+---------------------------------+-------------------+-----------+--------------+--------------+--------+
|artistName|                        trackName|          listen_ts|listen_hour|listen_weekday|minutes_played|is_night|
+----------+---------------------------------+-------------------+-----------+--------------+--------------+--------+
|Guy2Bezbar|          Belly (feat. Rick Ross)|2025-03-29 17:55:00|         17|             7|           1.1|   false|
|       L2B|                          Pélican|2025-03-30 17:55:00|         17|             1|          2.76|   false|
|     Dadju|               Donne-moi l’accord|2025-03-30 17:58:00|         17|             1|          3.11|   false|
|    Alonzo|TOUT VA BIEN (feat. Ninho & Naps)|2025-03-30 18:01:00|         18|             1|          3.22|   false|
|      Zola| TOUTE LA JOURNÉE (feat. Tiakola)|2025-03-30 18:04:00|         18|             1|           2.8|   false|
|  Coldplay|          

## 4. Streaming History — Exploration après nettoyage

In [6]:
print("=== Top 20 artistes les plus écoutés ===")
df_streams.groupBy("artistName") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 1).alias("total_minutes")
    ) \
    .orderBy(F.desc("nb_ecoutes")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Top 15 titres les plus écoutés ===")
df_streams.groupBy("artistName", "trackName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show(truncate=40)

=== Top 20 artistes les plus écoutés ===


+------------+----------+-------------+
|  artistName|nb_ecoutes|total_minutes|
+------------+----------+-------------+
|       Damso|       821|       2452.2|
|     Tiakola|       610|       1850.2|
|       Ninho|       566|       1639.0|
|     Zamdane|       476|       1462.6|
|  Guy2Bezbar|       469|       1075.5|
|Travis Scott|       440|       1419.8|
| La Mano 1.9|       384|        943.3|
|        Gazo|       366|       1007.1|
|   Bad Bunny|       340|        983.6|
|       Niska|       293|        763.2|
|        GIMS|       290|        733.9|
|        Ziak|       275|        683.8|
|     KAROL G|       273|        715.0|
|         MHD|       272|        761.1|
|        Leto|       267|        688.3|
|        Vald|       246|        546.7|
|     KeBlack|       243|        615.5|
|        Dave|       239|        920.4|
|      Kalash|       236|        669.8|
|     Jok'air|       225|        724.1|
+------------+----------+-------------+


=== Top 15 titres les plus écoutés ===

+-----------+----------------------------+-----+
| artistName|                   trackName|count|
+-----------+----------------------------+-----+
|      Asake|              BADMAN GANGSTA|  108|
|    kulturr|                        Busy|   94|
| Shallipopi|                     Laho II|   92|
|       OBOY|                    Jolie Go|   78|
|La Mano 1.9|            Crime ensoleillé|   78|
|       Gazo|     KAT (feat. La Rvfleuze)|   75|
|    Tiakola|                   PONA NINI|   73|
|    Tiakola|               PSYCHOLOGIQUE|   72|
| Guy2Bezbar|Jerrican (feat. La Mano 1.9)|   71|
|      Niska|                     Adriano|   71|
|La Mano 1.9|                 Businessman|   66|
|    Tiakola|                    NO LIMIT|   65|
|      Damso|                   Pa Pa Paw|   64|
|   Anyme023|                   Shamballa|   63|
|        L2B|                     Pélican|   61|
+-----------+----------------------------+-----+



In [7]:
print("=== Écoutes par heure de la journée ===")
df_streams.groupBy("listen_hour") \
    .count() \
    .orderBy("listen_hour") \
    .show(24)

print("\n=== Écoutes par jour de la semaine (1=dim, 2=lun, ..., 7=sam) ===")
df_streams.groupBy("listen_weekday") \
    .count() \
    .orderBy("listen_weekday") \
    .show()

print("\n=== % écoutes nocturnes (22h–5h) ===")
total = df_streams.count()
night = df_streams.filter(F.col("is_night")).count()
print(f"Nuit : {night:,} / {total:,} ({night/total*100:.1f}%)")

=== Écoutes par heure de la journée ===
+-----------+-----+
|listen_hour|count|
+-----------+-----+
|          0| 1487|
|          1| 1382|
|          2| 1193|
|          3|  790|
|          4|  512|
|          5|  885|
|          6| 1010|
|          7|  822|
|          8| 1072|
|          9| 1218|
|         10| 1382|
|         11| 1529|
|         12| 1721|
|         13| 1908|
|         14| 1820|
|         15| 1820|
|         16| 1768|
|         17| 1841|
|         18| 2071|
|         19| 1925|
|         20| 1651|
|         21| 1201|
|         22| 1397|
|         23| 1567|
+-----------+-----+


=== Écoutes par jour de la semaine (1=dim, 2=lun, ..., 7=sam) ===
+--------------+-----+
|listen_weekday|count|
+--------------+-----+
|             1| 4959|
|             2| 4891|
|             3| 4681|
|             4| 5046|
|             5| 4543|
|             6| 5178|
|             7| 4674|
+--------------+-----+


=== % écoutes nocturnes (22h–5h) ===
Nuit : 9,213 / 33,972 (27.1%)


In [8]:
print("=== Activité mensuelle ===")
df_streams.groupBy("listen_month") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 0).alias("total_minutes")
    ) \
    .orderBy("listen_month") \
    .show(20)

print("\n=== Semaines les plus actives ===")
df_streams.groupBy("listen_year", "listen_week") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Activité mensuelle ===
+------------+----------+-------------+
|listen_month|nb_ecoutes|total_minutes|
+------------+----------+-------------+
|     2025-03|       181|        387.0|
|     2025-04|      3123|       8067.0|
|     2025-05|      3596|       9753.0|
|     2025-06|      4071|      11521.0|
|     2025-07|      3697|      10126.0|
|     2025-08|      3828|      10366.0|
|     2025-09|      2404|       6511.0|
|     2025-10|      1868|       4804.0|
|     2025-11|      1821|       4504.0|
|     2025-12|      2888|       7651.0|
|     2026-01|      2920|       7790.0|
|     2026-02|      1663|       4257.0|
|     2026-03|      1912|       5075.0|
+------------+----------+-------------+


=== Semaines les plus actives ===
+-----------+-----------+-----+
|listen_year|listen_week|count|
+-----------+-----------+-----+
|       2025|         33| 1184|
|       2025|         26| 1153|
|       2025|         29| 1036|
|       2025|         25| 1028|
|       2025|         22|  945|
|

## 5. Schéma final Streaming History & écriture Parquet

In [9]:
df_streams_final = df_streams.select(
    F.col("artistName"),
    F.col("trackName"),
    F.col("msPlayed"),
    F.col("minutes_played"),
    F.col("listen_ts"),
    F.col("listen_year"),
    F.col("listen_month"),
    F.col("listen_hour"),
    F.col("listen_weekday"),
    F.col("listen_week"),
    F.col("is_night"),
    F.col("interaction_weight")
)

print(f"Lignes finales : {df_streams_final.count():,}")
df_streams_final.printSchema()

df_streams_final.write.mode("overwrite").parquet(OUT_STREAMS)
print(f"\n✓ Parquet écrit : {OUT_STREAMS}")

df_check = spark.read.parquet(OUT_STREAMS)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=40)

Lignes finales : 33,972
root
 |-- artistName: string (nullable = true)
 |-- trackName: string (nullable = true)
 |-- msPlayed: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- listen_ts: timestamp (nullable = true)
 |-- listen_year: integer (nullable = true)
 |-- listen_month: string (nullable = true)
 |-- listen_hour: integer (nullable = true)
 |-- listen_weekday: integer (nullable = true)
 |-- listen_week: integer (nullable = true)
 |-- is_night: boolean (nullable = false)
 |-- interaction_weight: double (nullable = false)




✓ Parquet écrit : ../../data/parquet/spotify_streams.parquet
✓ Vérification lecture : 33,972 lignes
+----------+-----------------+--------+--------------+-------------------+-----------+------------+-----------+--------------+-----------+--------+------------------+
|artistName|        trackName|msPlayed|minutes_played|          listen_ts|listen_year|listen_month|listen_hour|listen_weekday|listen_week|is_night|interaction_weight|
+----------+-----------------+--------+--------------+-------------------+-----------+------------+-----------+--------------+-----------+--------+------------------+
|   Chiktay|    La pli si tol|   97872|          1.63|2025-08-19 19:54:00|       2025|     2025-08|         19|             3|         34|   false|               1.0|
|Shallipopi|          Laho II|   81502|          1.36|2025-08-19 19:55:00|       2025|     2025-08|         19|             3|         34|   false|               1.0|
|  DJ Taffy|Peep peep pop pop|   31207|          0.52|2025-08-19

---
## 6. YourLibrary — Ingestion

Les titres sauvegardés = ce qui me représente le plus musicalement.  
Signal ALS fort → **poids 3.0** (vs 1.0 pour les streams ordinaires).

In [10]:
df_lib_raw = spark.read \
    .option("multiLine", "true") \
    .json(LIBRARY_PATH)

df_lib_raw.printSchema()

root
 |-- albums: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album: string (nullable = true)
 |    |    |-- artist: string (nullable = true)
 |    |    |-- uri: string (nullable = true)
 |-- artists: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- uri: string (nullable = true)
 |-- bannedArtists: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- bannedTracks: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- episodes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- kallaxes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- other: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- pageMatch: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- shows: array (nullable = true)
 |    |-- element: string (c

In [11]:
# Explosion du tableau 'tracks'
df_library = df_lib_raw \
    .select(F.explode("tracks").alias("t")) \
    .select(
        F.col("t.artist").alias("artistName"),
        F.col("t.album").alias("albumName"),
        F.col("t.track").alias("trackName"),
        F.col("t.uri").alias("trackUri")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(3.0))  # Signal fort : titre sauvegardé

print(f"Titres dans la bibliothèque : {df_library.count():,}")
df_library.show(10, truncate=45)

Titres dans la bibliothèque : 80
+-------------------+--------------------------------------+----------------------+------------------------------------+------------------+
|         artistName|                             albumName|             trackName|                            trackUri|interaction_weight|
+-------------------+--------------------------------------+----------------------+------------------------------------+------------------+
|            Fanny J|                       Vous les hommes|     Ancrée à ton port|spotify:track:05xLOpipHWbGG0CboVVrXy|               3.0|
|          I Am Roze|                Dollar - A COLORS SHOW|Dollar - A COLORS SHOW|spotify:track:0jMSZtSikKIovVEbss39FL|               3.0|
|               Dave|      We're All Alone In This Together|       We're All Alone|spotify:track:22agj1ppHejR41cUyf7k6v|               3.0|
|          Bad Bunny|                  DeBÍ TiRAR MáS FOToS|                BOKeTE|spotify:track:79x7xtoCLpf6l32Zz6mWo4|       

## 7. YourLibrary — Exploration

In [12]:
print("=== Top artistes dans la bibliothèque ===")
df_library.groupBy("artistName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Artistes présents dans la bibliothèque mais pas dans les streams ===")
lib_artists   = df_library.select("artistName").distinct()
stream_artists = df_streams.select("artistName").distinct()
only_in_lib = lib_artists.subtract(stream_artists)
print(f"Artistes sauvegardés jamais streamés : {only_in_lib.count():,}")
only_in_lib.limit(15).show(truncate=40)

=== Top artistes dans la bibliothèque ===
+-------------------+-----+
|         artistName|count|
+-------------------+-----+
|             Rijade|    7|
|        La Rvfleuze|    5|
|           Youv Dee|    3|
|               Dave|    2|
|       Travis Scott|    2|
|              Khali|    2|
|             Laylow|    2|
|            Orelsan|    2|
|               Naza|    2|
|         Juice WRLD|    1|
|        Beach House|    1|
|    Wallace Cleaver|    1|
|             Luidji|    1|
|                KIK|    1|
|Ngiah Tax Olo Fotsy|    1|
|       XXXTENTACION|    1|
|            Squidji|    1|
|            Ronisia|    1|
|  Empire Of The Sun|    1|
|        Chris Isaak|    1|
+-------------------+-----+


=== Artistes présents dans la bibliothèque mais pas dans les streams ===


Artistes sauvegardés jamais streamés : 3


+----------+
|artistName|
+----------+
|  Sico Vox|
|      BoBo|
|       TxC|
+----------+



## 8. YourLibrary — Écriture Parquet

In [13]:
df_library.write.mode("overwrite").parquet(OUT_LIBRARY)
print(f"✓ Parquet écrit : {OUT_LIBRARY}")

df_check = spark.read.parquet(OUT_LIBRARY)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

✓ Parquet écrit : ../../data/parquet/spotify_library.parquet
✓ Vérification lecture : 80 lignes
+----------+--------------------------------+----------------------+------------------------------------+------------------+
|artistName|                       albumName|             trackName|                            trackUri|interaction_weight|
+----------+--------------------------------+----------------------+------------------------------------+------------------+
|   Fanny J|                 Vous les hommes|     Ancrée à ton port|spotify:track:05xLOpipHWbGG0CboVVrXy|               3.0|
| I Am Roze|          Dollar - A COLORS SHOW|Dollar - A COLORS SHOW|spotify:track:0jMSZtSikKIovVEbss39FL|               3.0|
|      Dave|We're All Alone In This Together|       We're All Alone|spotify:track:22agj1ppHejR41cUyf7k6v|               3.0|
+----------+--------------------------------+----------------------+------------------------------------+------------------+
only showing top 3 rows



---
## 9. Playlist1 — Ingestion

31 playlists personnelles. Chaque track ajouté manuellement = curation active.  
Signal ALS intermédiaire → **poids 2.0**.

In [14]:
df_pl_raw = spark.read \
    .option("multiLine", "true") \
    .json(PLAYLIST_PATH)

df_pl_raw.printSchema()

root
 |-- playlists: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- collaborators: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- addedDate: string (nullable = true)
 |    |    |    |    |-- localTrack: struct (nullable = true)
 |    |    |    |    |    |-- uri: string (nullable = true)
 |    |    |    |    |-- track: struct (nullable = true)
 |    |    |    |    |    |-- albumName: string (nullable = true)
 |    |    |    |    |    |-- artistName: string (nullable = true)
 |    |    |    |    |    |-- trackName: string (nullable = true)
 |    |    |    |    |    |-- trackUri: string (nullable = true)
 |    |    |-- lastModifiedDate: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- numberOfFollowe

In [15]:
# Explosion playlists → items → track
df_playlists = df_pl_raw \
    .select(F.explode("playlists").alias("pl")) \
    .select(
        F.col("pl.name").alias("playlistName"),
        F.col("pl.lastModifiedDate").alias("lastModifiedDate"),
        F.explode("pl.items").alias("item")
    ) \
    .select(
        F.col("playlistName"),
        F.col("lastModifiedDate"),
        F.col("item.track.trackName").alias("trackName"),
        F.col("item.track.artistName").alias("artistName"),
        F.col("item.track.albumName").alias("albumName"),
        F.col("item.track.trackUri").alias("trackUri"),
        F.to_date(F.col("item.addedDate"), "yyyy-MM-dd").alias("addedDate")
    ) \
    .filter(F.col("trackName").isNotNull()) \
    .withColumn("interaction_weight", F.lit(2.0))  # Curation manuelle

print(f"Tracks totaux dans les playlists : {df_playlists.count():,}")
df_playlists.show(8, truncate=40)

Tracks totaux dans les playlists : 6,475
+------------+----------------+---------------------------+-----------+---------------------------+------------------------------------+----------+------------------+
|playlistName|lastModifiedDate|                  trackName| artistName|                  albumName|                            trackUri| addedDate|interaction_weight|
+------------+----------------+---------------------------+-----------+---------------------------+------------------------------------+----------+------------------+
|   LaZone 🪐|      2026-03-29|              DICTIONNAIRES|      Ninho|                  M.I.L.S 4|spotify:track:1xytC7yhTxk4ee5MIcIgGg|2026-01-08|               2.0|
|   LaZone 🪐|      2026-03-29|                  STOCKHOLM|      Ninho|                  M.I.L.S 4|spotify:track:4mqYjoQ5H9OX50hyxAakgd|2026-01-16|               2.0|
|   LaZone 🪐|      2026-03-29|                     Vision|  La Fouine|                     Vision|spotify:track:69WMiCz57nWMET

## 10. Playlists — Exploration

In [16]:
print("=== Toutes les playlists (nom + nb tracks) ===")
df_playlists.groupBy("playlistName", "lastModifiedDate") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(50, truncate=50)

print("\n=== Top artistes toutes playlists confondues ===")
df_playlists.groupBy("artistName") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=35)

=== Toutes les playlists (nom + nb tracks) ===
+----------------------+----------------+-----+
|          playlistName|lastModifiedDate|count|
+----------------------+----------------+-----+
|      Fast&Furious 🏎️|      2026-03-29|  728|
|              PGE 2025|      2025-07-23|  425|
|            Guapman 💷|      2026-03-13|  384|
|                Ski 🎿|      2026-03-25|  373|
|              Rhéto 🎓|      2025-12-05|  357|
|               Vibe 🌊|      2026-03-30|  301|
|           Vacation 🎒|      2026-03-15|  296|
|          Guayabo 🇪🇸|      2026-03-30|  272|
|          MADA CLUB 🧨|      2026-03-30|  269|
|             Angels 🦋|      2026-03-26|  255|
|              Antes 🍹|      2026-03-26|  216|
|            Rooftop 🏠|      2026-03-19|  215|
|       SpaceCookies 🍪|      2026-03-25|  205|
|           ThankGod 🪽|      2026-03-15|  180|
|           Lovin'it ⭐️|      2026-02-11|  178|
|         Adventure 🏔️|      2026-03-12|  157|
|           Aesthetic🍷|      2026-03-28|  149|
|        

In [17]:
print("=== Artistes présents dans les 3 sources (streams + library + playlists) ===")
pl_artists  = df_playlists.select("artistName").distinct()
lib_artists = df_library.select("artistName").distinct()
str_artists = df_streams.select("artistName").distinct()

core_artists = pl_artists.intersect(lib_artists).intersect(str_artists)
print(f"Artistes présents partout (noyau dur) : {core_artists.count():,}")
core_artists.limit(20).show(truncate=40)

=== Artistes présents dans les 3 sources (streams + library + playlists) ===
Artistes présents partout (noyau dur) : 54
+-------------------+
|         artistName|
+-------------------+
|             Rijade|
|        Beach House|
|         Juice WRLD|
|    Wallace Cleaver|
|             Luidji|
|                KIK|
|Ngiah Tax Olo Fotsy|
|            Squidji|
|       XXXTENTACION|
|            Ronisia|
|  Empire Of The Sun|
|        Chris Isaak|
|               OBOY|
|           Youv Dee|
|               Dave|
|       Travis Scott|
|                Djo|
|            Fanny J|
|     menace Santana|
|              HOUDI|
+-------------------+



## 11. Playlists — Écriture Parquet

In [18]:
df_playlists.write.mode("overwrite").parquet(OUT_PLAYLISTS)
print(f"✓ Parquet écrit : {OUT_PLAYLISTS}")

df_check = spark.read.parquet(OUT_PLAYLISTS)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.show(3, truncate=40)

✓ Parquet écrit : ../../data/parquet/spotify_playlists.parquet
✓ Vérification lecture : 6,475 lignes
+------------+----------------+-------------+----------+---------+------------------------------------+----------+------------------+
|playlistName|lastModifiedDate|    trackName|artistName|albumName|                            trackUri| addedDate|interaction_weight|
+------------+----------------+-------------+----------+---------+------------------------------------+----------+------------------+
|   LaZone 🪐|      2026-03-29|DICTIONNAIRES|     Ninho|M.I.L.S 4|spotify:track:1xytC7yhTxk4ee5MIcIgGg|2026-01-08|               2.0|
|   LaZone 🪐|      2026-03-29|    STOCKHOLM|     Ninho|M.I.L.S 4|spotify:track:4mqYjoQ5H9OX50hyxAakgd|2026-01-16|               2.0|
|   LaZone 🪐|      2026-03-29|       Vision| La Fouine|   Vision|spotify:track:69WMiCz57nWMET9vtwaKBf|2026-01-16|               2.0|
+------------+----------------+-------------+----------+---------+--------------------------------

---
## 12. YourSoundCapsule — Ingestion

Snapshots hebdomadaires récents. Seule source qui fournit des **données de genre** — absentes de `StreamingHistory`.  
Chaque entrée = une semaine avec `streamCount`, `secondsPlayed`, `topTracks`, `topArtists`, `topGenres`.

In [3]:
df_sc_raw = spark.read \
    .option("multiLine", "true") \
    .json(SOUND_CAPSULE_PATH)

df_sc_raw.printSchema()
print(f"Snapshots disponibles : {df_sc_raw.select(F.explode('stats')).count():,}")

root
 |-- highlights: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- date: string (nullable = true)
 |    |    |-- fansLikeYouHighlight: struct (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |    |-- entity: string (nullable = true)
 |    |    |    |-- numberOfListeners: long (nullable = true)
 |    |    |    |-- position: long (nullable = true)
 |    |    |    |-- previousPosition: long (nullable = true)
 |    |    |-- firstToDiscoverHighlight: struct (nullable = true)
 |    |    |    |-- country: string (nullable = true)
 |    |    |    |-- entity: string (nullable = true)
 |    |    |    |-- position: long (nullable = true)
 |    |    |-- highlightType: string (nullable = true)
 |    |    |-- multiEntityMilestoneHighlight: struct (nullable = true)
 |    |    |    |-- entities: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- milestoneListeningSeconds:

In [4]:
# Explosion stats → une ligne par snapshot hebdomadaire
df_sc = df_sc_raw \
    .select(F.explode("stats").alias("s")) \
    .select(
        F.to_date(F.col("s.date"), "yyyy-MM-dd").alias("snapshot_date"),
        F.col("s.streamCount").alias("stream_count"),
        F.round(F.col("s.secondsPlayed") / 60.0, 1).alias("minutes_played"),
        F.col("s.topGenres").alias("top_genres"),     # array<string>
        F.col("s.topArtists").alias("top_artists"),   # array<struct>
        F.col("s.topTracks").alias("top_tracks")      # array<struct>
    ) \
    .filter(F.col("snapshot_date").isNotNull()) \
    .orderBy("snapshot_date")

print(f"Snapshots après nettoyage : {df_sc.count():,}")
df_sc.select("snapshot_date", "stream_count", "minutes_played", "top_genres").show(10, truncate=60)

Snapshots après nettoyage : 3
+-------------+------------+--------------+----------+
|snapshot_date|stream_count|minutes_played|top_genres|
+-------------+------------+--------------+----------+
|   2026-03-09|           0|           0.0|        []|
|   2026-03-16|           0|           0.0|        []|
|   2026-03-23|           0|           0.0|        []|
+-------------+------------+--------------+----------+



## 13. YourSoundCapsule — Exploration

In [5]:
print("=== Activité hebdomadaire (stream_count + minutes) ===")
df_sc.select("snapshot_date", "stream_count", "minutes_played").show(20)

print("\n=== Genres les plus fréquents (toutes semaines confondues) ===")
df_sc.select(F.explode("top_genres").alias("genre")) \
    .groupBy("genre") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show(truncate=40)

print("\n=== Top artistes par semaine ===")
df_sc.select(
    F.col("snapshot_date"),
    F.explode("top_artists").alias("a")
).select(
    "snapshot_date",
    F.col("a.name").alias("artistName"),
    F.col("a.streamCount").alias("streams_semaine")
).orderBy("snapshot_date", F.desc("streams_semaine")) \
 .show(20, truncate=35)

=== Activité hebdomadaire (stream_count + minutes) ===
+-------------+------------+--------------+
|snapshot_date|stream_count|minutes_played|
+-------------+------------+--------------+
|   2026-03-09|           0|           0.0|
|   2026-03-16|           0|           0.0|
|   2026-03-23|           0|           0.0|
+-------------+------------+--------------+


=== Genres les plus fréquents (toutes semaines confondues) ===
+-----+-----+
|genre|count|
+-----+-----+
+-----+-----+


=== Top artistes par semaine ===
+-------------+---------------+---------------+
|snapshot_date|     artistName|streams_semaine|
+-------------+---------------+---------------+
|   2026-03-09|        La Fève|             37|
|   2026-03-09|            PLK|             29|
|   2026-03-09|   Travis Scott|             24|
|   2026-03-09|    Jolagreen23|             19|
|   2026-03-09|  Playboi Carti|             13|
|   2026-03-09|   Metro Boomin|             12|
|   2026-03-09|      21 Savage|             11|
|

## 14. YourSoundCapsule — Écriture Parquet

In [6]:
df_sc.write.mode("overwrite").parquet(OUT_SOUND_CAPSULE)
print(f"✓ Parquet écrit : {OUT_SOUND_CAPSULE}")

df_check = spark.read.parquet(OUT_SOUND_CAPSULE)
print(f"✓ Vérification lecture : {df_check.count():,} lignes")
df_check.select("snapshot_date", "stream_count", "minutes_played", "top_genres").show(3, truncate=60)

✓ Parquet écrit : ../../data/parquet/spotify_sound_capsule.parquet
✓ Vérification lecture : 3 lignes
+-------------+------------+--------------+----------+
|snapshot_date|stream_count|minutes_played|top_genres|
+-------------+------------+--------------+----------+
|   2026-03-09|           0|           0.0|        []|
|   2026-03-16|           0|           0.0|        []|
|   2026-03-23|           0|           0.0|        []|
+-------------+------------+--------------+----------+



---
## 15. Résumé global

| Source | Contenu | Output | Usage ML |
|---|---|---|---|
| `StreamingHistory_music_0-7.json` | ~95k écoutes filtrées >30s | `spotify_streams.parquet` | ALS (poids 1) + K-Means (heure/jour) |
| `YourLibrary.json` | ~1 500 titres sauvegardés | `spotify_library.parquet` | ALS (poids 3) |
| `Playlist1.json` | 31 playlists, curation manuelle | `spotify_playlists.parquet` | ALS (poids 2) |
| `YourSoundCapsule.json` | Snapshots hebdo + genres | `spotify_sound_capsule.parquet` | K-Means (genre par semaine) |

**Prochaines étapes :**
- **Notebook ALS** : unifier les 3 sources interactions → matrice `(user, track_id, weight)` → `interactions.parquet`
- **Notebook K-Means** : combiner `listen_hour`, `listen_weekday`, `is_night`, `minutes_played` (streams) + `top_genres` (sound capsule) → `activity_vectors.parquet`

In [7]:
spark.stop()